# Kerr Circular EMRIs in Accretion Disks — Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stephengreen/DirtyEMRI-tutorial/blob/main/Playground_KerrCircularAccretion.ipynb)

## Setup

**On Google Colab:** run the **first code cell** below. It installs `conda` and then
**restarts the runtime — this is expected and normal.** After it restarts, choose
**Runtime → Run all** (the setup picks up where it left off, builds the model in ~2-3
minutes, and the physics cells run).

**Locally (via `pixi run lab`):** the two setup cells below are a no-op — the build
already happened. Just run the notebook top to bottom.


In [ ]:
# === Colab setup 1/2: install conda. THIS RESTARTS THE RUNTIME (expected). ===
# After the restart, just run 'Runtime -> Run all' again: this cell sees conda is
# already installed and skips straight through (no second restart). No-op locally.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("/usr/local/conda-meta"):
    !pip install -q condacolab
    import condacolab
    condacolab.install()        # installs Miniforge to /usr/local, then RESTARTS the kernel


In [ ]:
# === Colab setup 2/2: fetch the code, install the (tested) deps, build the fork ===
import sys, os, pathlib
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/stephengreen/DirtyEMRI-tutorial.git"
    if not os.path.exists("/content/DirtyEMRI-tutorial"):
        !git clone --depth 1 $REPO_URL /content/DirtyEMRI-tutorial
    os.chdir("/content/DirtyEMRI-tutorial")
    # condacolab pins python to Colab's version (3.12) but its base is 3.11, so mamba
    # refuses to install. Drop the stale pin so the solve proceeds on the base python.
    !rm -f /usr/local/conda-meta/pinned
    # conda gives the exact recipe proven locally: gsl 2.7 (not 2.8) and a real libhdf5.so
    !mamba install -q -y -c conda-forge "numpy<2" "cython<3" scipy "gsl=2.7" lapack liblapacke openblas hdf5 h5py requests tqdm matplotlib-base
    os.environ["CONDA_PREFIX"] = "/usr/local"   # so build_few.py uses conda's gsl/hdf5 + rpath
    !python build_few.py

# Make `import few` (and its compiled extensions) resolve here — works for the Colab
# conda base env and for the local pixi env alike.
_few_dir = pathlib.Path.cwd() / "FastEMRIWaveforms"
if _few_dir.exists() and str(_few_dir) not in sys.path:
    sys.path.insert(0, str(_few_dir))

from few.trajectory.inspiral import EMRIInspiral  # sanity check
print("FastEMRIWaveforms ready - KerrCircFlux available.")


# Playground for Kerr Circular Orbits
In this note book you can explore how Kerr Circular Orbits work. An extensive tutorial about the usage of the trajectories can be found [here](https://github.com/lorenzsp/GRAPPA_EMRI_tutorial/blob/main/EMRI_Waveforms_in_a_nutshell.ipynb). As a starting point you should try to explore how much

In [ ]:
import numpy as np
import warnings

from few.trajectory.inspiral import EMRIInspiral
from few.amplitude.romannet import RomanAmplitude
from few.amplitude.interp2dcubicspline import Interp2DAmplitude
from few.waveform import FastSchwarzschildEccentricFlux, SlowSchwarzschildEccentricFlux
from few.utils.utility import get_overlap, get_mismatch, get_separatrix, get_fundamental_frequencies, get_fundamental_frequencies_spin_corrections
from few.utils.ylm import GetYlms
from few.utils.modeselector import ModeSelector
from few.summation.interpolatedmodesum import CubicSplineInterpolant
from few.utils.constants import *
import matplotlib.pyplot as plt


try:
    import cupy as xp

    gpu_available = True

except (ModuleNotFoundError, ImportError) as e:
    import numpy as xp

    warnings.warn(
        "CuPy is not installed or a gpu is not available. If trying to run on a gpu, please install CuPy."
    )
    gpu_available = False

traj = EMRIInspiral(func="KerrCircFlux")

In [ ]:
# parameter of https://arxiv.org/pdf/2207.10086.pdf
# set initial parameters
M = 1e6
mu = 50.0
p0 = 15.482608237080893
e0 = 0.0
a = 0.9
x0 = 1.0
# accretion parameters from https://arxiv.org/abs/2207.10086
A = 1e-5
n = 8.0
m = 0.0

# run trajectory
T = 4.0 # years
dt = 10.0
phiphi0, phitheta, phir = 0, 0, 0

# without accretion
t, p, e, x, Phi_phi, Phi_theta, Phi_r = traj(M, mu, a, p0, e0, x0, 0.0, n, m, T=T, dt=dt)
tfinal = t[-1]
spline_noAcc = CubicSplineInterpolant(t, np.stack((p, e, x, Phi_phi, Phi_theta, Phi_r)) )

plt.figure()
# with accretion
Avec = 10**np.linspace(-7,-3,num=5)
for A in Avec:
    t, p, e, x, Phi_phi, Phi_theta, Phi_r = traj(M, mu, a, p0, e0, x0, A, n, m, T=T, dt=dt)
    spline = CubicSplineInterpolant(t, np.stack((p, e, x, Phi_phi, Phi_theta, Phi_r)) )
    plt.semilogy(t/(YRSID_SI), np.abs(spline(t)[3]-spline_noAcc(t)[3]),'--',label=f'A={A:.2e}')

plt.legend()
plt.xlabel('t [ry]')
plt.ylim([1e-1,1e5])
plt.grid()
plt.ylabel('Phase difference')

plt.figure()
# with accretion
Avec = 10**np.linspace(-7,-4,num=5)
for A in Avec:
    t, p, e, x, Phi_phi, Phi_theta, Phi_r = traj(M, mu, a, p0, e0, x0, A, n, m, T=T, dt=dt)
    spline = CubicSplineInterpolant(t, np.stack((p, e, x, Phi_phi, Phi_theta, Phi_r)) )
    f = get_fundamental_frequencies(a, *spline(t)[:3])[0] / (2*np.pi*M*MTSUN_SI)
    plt.loglog(f, np.abs(spline(t)[3]-spline_noAcc(t)[3]),'--',label=f'A={A:.2e}')

plt.axhline(1, color='k')
plt.legend()
plt.xlabel('f [Hz]')
plt.ylim([1e-1,1e5])
plt.grid()
plt.ylabel('relative difference in phase')


In [ ]:
plt.figure()
for n,m,lintype in zip([8.0, 5.9], [0.0, 0.0], ['-', '-.']):#[-28/5, -12.0]
    deph = []
    Avec = np.linspace(0.1e-7,2e-4,num=30)
    for A in Avec:
        # with accretion
        t, p, e, x, Phi_phi, Phi_theta, Phi_r = traj(M, mu, a, p0, e0, x0, A, n, m, T=T, dt=dt)
        spline = CubicSplineInterpolant(t, np.stack((p, e, x, Phi_phi, Phi_theta, Phi_r)) )
        deph.append(np.max(np.abs(spline(t)[3]-spline_noAcc(t)[3])) )

    plt.loglog(Avec, deph,lintype,label=f'n={n},m={m}')

plt.legend()
plt.axhline(1.0,linestyle='--',color='k')
plt.axhline(100.0,linestyle='--',color='k')
plt.xlabel('A')
# plt.ylim([1e-1,1e3])
plt.grid(True, which="both")
plt.ylabel('Phase difference at the end of the trahectory')
# similat to the phase difference of Fig 1 https://arxiv.org/pdf/2207.10086.pdf

In [ ]:
fig, axes = plt.subplots(2, 3)
plt.subplots_adjust(wspace=0.3)
fig.set_size_inches(14, 8)
axes = axes.ravel()

ylabels = [r'$e$', r'$p$', r'$e$', r'$\Phi_\phi$', r'$\Phi_r$']
xlabels = [r'$p$', r'$t$', r'$t$', r'$t$', r'$t$', r'$t$', r'$t$']
ys = [e, p, e, Phi_phi, Phi_r]
xs = [p, t, t, t, t]

for i, (ax, x, y, xlab, ylab) in enumerate(zip(axes, xs, ys, xlabels, ylabels)):
    ax.plot(x, y)
    ax.set_xlabel(xlab, fontsize=16)
    ax.set_ylabel(ylab, fontsize=16)

In [ ]:
plt.figure()
plt.semilogy(t, np.abs(spline(t)[3]-spline_noAcc(t)[3]),'--')
plt.xlabel('t')
plt.ylim([1e-1,1e4])
plt.grid()
plt.ylabel('Phase difference')
# similat to the phase difference of Fig 1 https://arxiv.org/pdf/2207.10086.pdf

In [ ]:
# to obtain the right hand side of the ODE you can use
pdot, edot, Ydot, OmegaPhi, OmegaTheta, OmegaR = traj.get_derivative(mu/M, a, p0, e0, x0, np.asarray([A, n, m]))

# range
pvec = np.linspace(3.0, 30.0,dtype=float)

# check that diff is small
diff = np.asarray([1-traj.get_derivative(mu/M, a, pp, 0.0, 1.0, np.asarray([A, n, m]))[3] /
get_fundamental_frequencies(np.ones(1)*a, np.ones(1)*pp, np.ones(1)*1e-10, np.ones(1)*1.0-1e-10)[0] 
# (1 / a + pp**(3/2))
for pp in pvec])

In [ ]:
A, n, m = 2e-5, 8.0, 0.0

# evolve sytem
pdot_evolution_Accretion = np.asarray([traj.get_derivative(mu/M, a, pp, 0.0, 1.0, np.asarray([A, n, m]))[0] for pp in pvec])
pdot_evolution_NoAccretion = np.asarray([traj.get_derivative(mu/M, a, pp, 0.0, 1.0, np.asarray([0.0, n, m]))[0] for pp in pvec])

def Power(x,a):
    return x**a
def Sqrt(x):
    return np.sqrt(x)

dL_dp = ((-3*Power(a,3) + Power(a,2)*(8 - 3*pvec)*Sqrt(pvec) + (-6 + pvec)*Power(pvec,2.5) + 3*a*pvec*(-2 + 3*pvec))/(2.*Power(2*a + (-3 + pvec)*Sqrt(pvec),1.5)*Power(pvec,1.75)))

LdotPN = -32./5. * mu/M *Power(pvec, -7./2.)

factor = LdotPN / dL_dp

plt.figure()
plt.loglog(pvec, (pdot_evolution_Accretion-pdot_evolution_NoAccretion) / (factor) )
# plt.loglog(pvec, pdot_evolution_NoAccretion/factor )
plt.loglog(pvec, A*(pvec/10.0)**(n),color='k',label=f'powerlaw model A={A}, n={n}', linestyle='--')
plt.ylabel('pdot due to Accretion')
plt.xlabel('p')
plt.grid()
plt.legend()
plt.show()

print("max diff",np.max(1-(pdot_evolution_Accretion-pdot_evolution_NoAccretion) / (factor) / ( A*(pvec/10.0)**(n)) ))

In [ ]:

plt.figure()
plt.loglog(pvec, -A*(pvec/10.0)**(n) * LdotPN/dL_dp, label='Accretion' )
plt.loglog(pvec, -pdot_evolution_NoAccretion, label='Adiabatic GW')
plt.loglog(pvec, -LdotPN/dL_dp, label='PN GW')

plt.ylabel('pdot')
plt.xlabel('p')
plt.grid()
plt.legend()
plt.show()